# Computational Banking with SciPy
## A Data Analytics Project — Bond Pricing, Portfolio Risk, ALM & Systemic Risk

*Based on Steinkamp, V., "Python for Engineering and Scientific Computing" (2024),
Chapter 6 — "Numerical Computations and Simulations Using SciPy".*

Every numerical technique in this notebook (root-finding, constrained
optimization, interpolation, differentiation, integration, ODEs, FFT) mirrors
an example from the source chapter, retargeted from engineering signals to
core banking / treasury / risk problems. Work through the notebook top to
bottom; each section has:

1. **Theory** (markdown) — the banking/finance model and the math behind it.
2. **Data** — a synthetic, reproducible dataset (fixed random seed).
3. **Task** — `TODO` cells for you to complete.
4. **Interpretation** — a short markdown cell where you explain the result.

Run the setup cell first.

## 0. Setup

Import the libraries used throughout the project. If `numdifftools` is not
installed, run `!pip install numdifftools --break-system-packages` in a cell.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import root, minimize
from scipy.interpolate import interp1d, CubicSpline
from scipy.integrate import quad, solve_ivp
from scipy.fft import fft, fftfreq
import numdifftools as nd

np.random.seed(7)          # reproducibility -- keep this seed unchanged
plt.rcParams["figure.figsize"] = (7, 4.5)
plt.rcParams["axes.grid"] = True

## 1. Solving for Yield to Maturity (Root-Finding)
### (mirrors book §6.1, `scipy.optimize.root`)

**Theory.** A bond pays semi-annual coupons of `F*c/2` and returns the face
value `F` at maturity `T`. Given a market price `P_market`, the **yield to
maturity (YTM)** `y` is the single discount rate that makes the present value
of all cash flows equal the market price:

```
P(y) = sum_{i=1}^{n} (F*c/m) / (1+y/m)^i  +  F / (1+y/m)^n  = P_market
```

where `m` = coupons per year, `n = T*m` = number of periods. `P(y)` cannot be
inverted algebraically for `y`, so — exactly like the damped-oscillation
zero-finding problem in Listing 6.1 — we solve `f(y) = P(y) - P_market = 0`
numerically with `scipy.optimize.root`.

In [ ]:
# ---- Bond terms (given) ----
F = 1000.0     # face value
c = 0.05       # annual coupon rate
T = 10         # years to maturity
m = 2          # coupon payments per year (semi-annual)
n = int(T*m)   # number of coupon periods

P_market = 950.0   # observed market price (trading at a discount to par)

def bond_price(y):
    # Present value of the bond cash flows at yield y (annualized, compounded m times/yr)
    coupon = F*c/m
    i = np.arange(1, n+1)
    pv_coupons = np.sum(coupon / (1+y/m)**i)
    pv_face = F / (1+y/m)**n
    return pv_coupons + pv_face

In [ ]:
# TODO 1a: Define f(y) = bond_price(y[0]) - P_market
def f(y):
    ...  # your code here

# TODO 1b: Use scipy.optimize.root(f, y0, method='hybr') starting from
#          y0 = [0.05] to solve for the YTM. Extract the scalar yield
#          into `ytm`.
y0 = [0.05]
ytm = None  # replace

print(f"Yield to maturity = {ytm}")

# TODO 1c: Compute the simple "current yield" = (F*c) / P_market for comparison.
current_yield = None  # replace

# TODO 1d: Plot the price-yield curve bond_price(y) over
#          y_grid = np.linspace(0.01, 0.12, 200), with a horizontal line at
#          P_market and the YTM solution point marked.

**Interpretation (answer here).** Is the YTM higher or lower than the coupon
rate `c = 5%`? Explain why that must be true given that the bond trades below
par (`P_market < F`).

## 2. Minimum-Variance Investment Portfolio (Constrained Optimization)
### (mirrors book §6.2, `scipy.optimize.minimize` — the cylinder-surface-area problem)

**Theory.** A bank's treasury desk holds an investment securities portfolio
across several asset classes. Given expected returns `mu` and a
covariance matrix `Sigma`, the classic **Markowitz minimum-variance
portfolio** solves:

```
minimize    w^T Sigma w                (portfolio variance)
subject to  sum(w) = 1                 (fully invested)
            w^T mu >= target_return    (meets a minimum yield requirement)
            w_i >= 0                   (no short selling)
```

This has exactly the same shape as the book's cylinder-minimization problem —
minimize an objective subject to an equality and inequality constraint — solved
with `scipy.optimize.minimize` using `method="SLSQP"`.

In [ ]:
# ---- Asset classes, expected annual returns, and volatilities (given) ----
assets = ["Treasuries", "MBS", "Corporate Bonds", "Municipal Bonds", "Equities"]
mu = np.array([0.025, 0.035, 0.045, 0.030, 0.080])          # expected returns
vols = np.array([0.020, 0.050, 0.070, 0.030, 0.180])        # volatilities

# correlation matrix between asset classes
corr = np.array([
    [ 1.00, 0.60, 0.20, 0.50, -0.05],
    [ 0.60, 1.00, 0.30, 0.40,  0.00],
    [ 0.20, 0.30, 1.00, 0.25,  0.35],
    [ 0.50, 0.40, 0.25, 1.00,  0.05],
    [-0.05, 0.00, 0.35, 0.05,  1.00],
])
Sigma = np.outer(vols, vols) * corr    # covariance matrix
target_return = 0.04                    # minimum required portfolio yield
n_assets = len(assets)

In [ ]:
# TODO 2a: Define portfolio_var(w) = w @ Sigma @ w
def portfolio_var(w):
    ...  # your code here

# TODO 2b: Define budget_constraint(w) = sum(w) - 1  (equality, must be 0)
def budget_constraint(w):
    ...  # your code here

# TODO 2c: Define return_constraint(w) = w @ mu - target_return (inequality, must be >= 0)
def return_constraint(w):
    ...  # your code here

# TODO 2d: Call scipy.optimize.minimize(portfolio_var, w0, method="SLSQP",
#          bounds=bounds, constraints=[...]) with both constraints above
#          and bounds = [(0,1)]*n_assets (no short-selling).
w0 = np.ones(n_assets) / n_assets
bounds = [(0, 1)] * n_assets
result = None  # replace
w_star = None  # replace with result.x

df = pd.DataFrame({"Asset": assets, "Weight": w_star})
print(df)

# TODO 2e: Plot the resulting weights as a bar chart (ax.bar(assets, w_star)).

# TODO 2f (bonus): Loop target_return over several values (e.g.
#          np.linspace(mu.min(), mu.max()*0.95, 25)), re-solve each time,
#          and plot portfolio volatility vs. target return to trace out
#          the efficient frontier.

**Interpretation (answer here).** Which asset class gets (close to) zero
weight in the minimum-variance solution, and why, given its correlation
profile with the other assets? What happens to the allocation if you raise
`target_return` closer to the equities-only return of 8%?

## 3. Building a Discount Curve to Value a Loan Book (Interpolation)
### (mirrors book §6.3, `scipy.interpolate.interp1d` — recovering a sampled signal)

**Theory.** Banks only observe interest rates at a handful of standard
tenors (overnight, 3-month, 1-year, 5-year swap, ...). To present-value a
loan with cash flows at *arbitrary* dates (e.g., a loan cash flow at month 17),
the bank must **interpolate** a continuous zero/discount curve from the
observed quotes — precisely the same "recover the continuous signal from
sampled points" problem as Listing 6.3.

Given a zero rate curve `r(t)`, the discount factor for a cash flow at time
`t` is `DF(t) = exp(-r(t) * t)`, and the present value of a loan's cash flows
is `PV = sum(CF_i * DF(t_i))`.

We compare **linear** and **cubic-spline** interpolation, and use the curve
to value a simple amortizing loan.

In [ ]:
# ---- Observed SOFR / swap market quotes at standard tenors ----
tenors = np.array([1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10])   # in years
zero_rates_pct = np.array([5.30, 5.25, 5.10, 4.80, 4.40, 4.20, 4.10, 4.15, 4.25])

df_curve = pd.DataFrame({"Tenor (yrs)": tenors, "Zero rate (%)": zero_rates_pct})
df_curve

# ---- A 3-year bullet loan making semi-annual interest + a final principal payment ----
loan_principal = 5_000_000.0
loan_coupon = 0.06     # loan's contractual rate
loan_tenors = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
loan_cashflows = np.full(6, loan_principal*loan_coupon/2)
loan_cashflows[-1] += loan_principal   # final payment includes principal

In [ ]:
# TODO 3a: Build a linear interpolant with interp1d(tenors, zero_rates_pct, kind="linear")
lin_curve = None  # replace

# TODO 3b: Build a cubic spline interpolant with CubicSpline(tenors, zero_rates_pct)
cubic_curve = None  # replace

# TODO 3c: Plot both curves over t_grid = np.linspace(1/12, 10, 400) together
#          with the original market quotes as scatter points.

# TODO 3d: Write discount_factor(t) = exp(-r(t)*t) using cubic_curve(t)/100
#          for r(t), then compute the present value of the loan's cash
#          flows: loan_pv = sum(loan_cashflows * discount_factor(loan_tenors))
def discount_factor(t):
    ...  # your code here

loan_pv = None  # replace
print(f"Present value of the loan = {loan_pv}")

**Interpretation (answer here).** The discount curve is downward-sloping at
the short end and roughly flat/upward after 2 years ("hump" shape). What does
that shape typically signal about market expectations for future short-term
rates (e.g., anticipated rate cuts)?

## 4. Interest-Rate Risk & Credit Risk (Differentiation & Integration)
### (mirrors book §6.4 `numdifftools.Derivative` and §6.5 `scipy.integrate.quad`)

**Theory — differentiation (interest-rate risk).** Using the same bond from
Section 1, **modified duration** measures how sensitive the bond's price is
to a small change in yield, and **convexity** captures the curvature of that
relationship:

```
Duration  D = -(1/P) * dP/dy
Convexity C =  (1/P) * d^2P/dy^2
```

exactly as the book differentiates a position function to get velocity
(Listing 6.10). We then use a **second-order Taylor approximation**
`ΔP/P ≈ -D·Δy + 0.5·C·Δy²` — the standard bond-risk approximation used on
every fixed-income trading desk — and check it against the exact repriced bond.

**Theory — integration (credit risk).** Under CECL/IFRS 9, a bank must
estimate **Expected Credit Loss (ECL)** over the life of a loan. Given a
default hazard rate `h(t)` (instantaneous default probability at time `t`) and
survival function `S(t) = exp(-∫₀^t h(u)du)`:

```
ECL = ∫₀^T EAD(t) * LGD * h(t) * S(t) dt
```

We use a Weibull hazard (common in credit-risk modeling for capturing rising
default risk over a loan's early life) and compute the integral with
`scipy.integrate.quad`, exactly as the book integrates to compute displacement
or work done.

In [ ]:
# ---- Duration/convexity: reuse the Section-1 bond and its YTM `ytm` ----
# (re-run Section 1 first so `bond_price` and `ytm` are defined)

# ---- Expected Credit Loss: Weibull hazard parameters (given) ----
k_shape, lam_scale = 1.5, 8.0    # Weibull hazard shape/scale (years)
EAD = 1_000_000.0                # exposure at default
LGD = 0.45                        # loss given default (45%)
T_loan = 5.0                      # loan life in years

def hazard(t):
    return (k_shape/lam_scale) * (t/lam_scale)**(k_shape - 1)

In [ ]:
# TODO 4a: Use numdifftools.Derivative(bond_price, n=1) and n=2 to build
#          dP_dy and d2P_dy2, evaluated at the Section-1 `ytm`.
dP_dy = None    # replace
d2P_dy2 = None  # replace

P0 = bond_price(ytm)
duration = None   # replace: -dP_dy(ytm) / P0
convexity = None  # replace: d2P_dy2(ytm) / P0
print(f"Modified duration = {duration}")
print(f"Convexity = {convexity}")

# TODO 4b: For a +100bp shock (dy=0.01), compute the Taylor approximation
#          -duration*dy + 0.5*convexity*dy**2 and compare it to the EXACT
#          %% price change (bond_price(ytm+dy)-P0)/P0.

# TODO 4c: Write cum_hazard(t) using scipy.integrate.quad(hazard, 0, t),
#          then survival(t) = exp(-cum_hazard(t)).
def cum_hazard(t):
    ...  # your code here

def survival(t):
    ...  # your code here

# TODO 4d: Write ecl_integrand(t) = EAD*LGD*hazard(t)*survival(t), then
#          integrate it from 0 to T_loan with scipy.integrate.quad to get
#          the total Expected Credit Loss.
def ecl_integrand(t):
    ...  # your code here

ECL = None  # replace
print(f"Expected Credit Loss = {ECL}")

# TODO 4e: Plot the survival curve and the ECL integrand (shaded area) side by side.

**Interpretation (answer here).** A bond with higher duration is riskier to
rate changes — is that consistent with what you found for this 10-year bond
vs. what you'd expect for a 2-year bond (qualitatively)? Separately, does the
Weibull hazard here mean default risk is rising or falling over the loan's
life, and how do you know from `k_shape`?

## 5. The Vasicek Mean-Reverting Short-Rate Model (ODE)
### (mirrors book §6.6, `scipy.integrate.solve_ivp` — spring-mass / population ODEs)

**Theory.** Bank treasury (ALM) desks model the short-term interest rate as
mean-reverting: it drifts back toward a long-run level `b` at speed `a`. The
deterministic drift component of the (Ornstein–Uhlenbeck / Vasicek) short-rate
model is:

```
dr/dt = a * (b - r)
```

This is structurally identical to the first-order linear ODEs solved with
`solve_ivp` in the book (e.g., Listing 6.16, cooling/decay-type equations),
and it has the same closed-form analytical solution as a damped exponential:
`r(t) = b + (r0 - b) * exp(-a*t)`, converging to the steady state `r* = b`
— useful for validating our numerical solver.

Banks use this model (or its full stochastic version, adding a `sigma*dW`
noise term) to simulate future rate paths for interest-rate risk stress
testing and to price interest-rate derivatives.

In [ ]:
a_speed = 0.4     # mean-reversion speed
b_level = 0.03     # long-run mean short rate (3%)
r0 = 0.01          # current short rate (1%)
t_span = (0, 15)
t_eval = np.linspace(*t_span, 300)

In [ ]:
# TODO 5a: Define vasicek(t, r) returning a_speed*(b_level - r[0])
def vasicek(t, r):
    ...  # your code here

# TODO 5b: Solve with solve_ivp(vasicek, t_span, [r0], t_eval=t_eval)
sol = None  # replace

# TODO 5c: Compute the analytical solution
#          analytic = b_level + (r0-b_level)*np.exp(-a_speed*sol.t)
#          and compare (max absolute difference) to the numerical solution
#          to validate solve_ivp.
analytic = None  # replace

# TODO 5d: Plot the simulated short-rate path together with a horizontal
#          dashed line at the long-run mean b_level.

# TODO 5e (bonus): Loop over a_speed in [0.1, 0.4, 1.0], resolve the ODE for
#          each, and plot all three paths together to see how mean-reversion
#          speed changes how quickly the rate approaches its long-run level.

**Interpretation (answer here).** If regulators require the bank to stress-test
its balance sheet against a scenario where rates *don't* revert quickly
(low `a`), how would that change the bank's estimated interest-rate risk over
a 5-year horizon compared to the base case?

## 6. Detecting Cyclicality in Loan Charge-Off Rates (Fourier Transform)
### (mirrors book §6.7 and the bearing-vibration example in §6.9 —
### `scipy.fft.fft`/`fftfreq` used to find hidden periodicities in a noisy signal)

**Theory.** Monthly loan charge-off (write-off) rates are noisy but often
contain **two overlapping cycles**: a multi-year credit/business cycle and a
12-month seasonal pattern (e.g., consumer credit card charge-offs are
seasonal). This is exactly the same signal-processing problem as the book's
bearing-vibration fault detection (Listing 6.32) — a hidden periodic signal
buried in noise — except here the two hidden frequencies are a business cycle
and a calendar-year seasonal cycle instead of two mechanical fault frequencies.

In [ ]:
# ---- Synthetic monthly charge-off rate (%) over 30 years = 360 months ----
n_months = 360
t_m = np.arange(n_months)
Ta = 1/12                      # sampling interval in years (monthly)

business_cycle_years = 7.0     # hidden business-cycle length
seasonal_period_years = 1.0    # hidden calendar-year seasonality

base_rate = 2.0
business_amp = 1.0
seasonal_amp = 0.5
noise = np.random.normal(0, 0.3, n_months)

charge_off_rate = (base_rate
                    + business_amp * np.sin(2*np.pi*t_m*Ta/business_cycle_years)
                    + seasonal_amp * np.sin(2*np.pi*t_m*Ta/seasonal_period_years)
                    + noise)

fig, ax = plt.subplots()
ax.plot(t_m*Ta, charge_off_rate)
ax.set(xlabel="Year", ylabel="Charge-off rate (%)", title="Synthetic Monthly Charge-Off Rate")
plt.show()

In [ ]:
# TODO 6a: De-mean the series: co_detrended = charge_off_rate - charge_off_rate.mean()
co_detrended = None  # replace

# TODO 6b: Compute the FFT with fft(co_detrended) and the frequency axis
#          with fftfreq(N, Ta) where N=n_months and Ta=1/12 (years/month).
CO_fft = None   # replace
freqs = None    # replace

# TODO 6c: Compute normalized amplitudes amps = 2*np.abs(CO_fft)/N, keep only
#          positive frequencies, and plot amplitude vs. frequency.

# TODO 6d: Find the TWO frequencies with the largest amplitude (excluding
#          freq=0), convert each to a period (1/frequency), and compare to
#          the true hidden periods (business_cycle_years=7.0, seasonal=1.0).
top2_freqs = None    # replace
top2_periods = None  # replace
print("Detected cycle periods (years):", top2_periods)

**Interpretation (answer here).** Why does it make business sense to
separate the seasonal component from the multi-year business-cycle component
before feeding a charge-off forecast into a loan-loss provisioning (CECL)
model? What would happen if you mistook the seasonal peak for the start of a
recession?

## 7. Bonus: Interbank Contagion / Systemic Risk (ODE System)
### (structurally the same ODE system as the book's epidemic model, §6.11.5,
### Listing 6.34 — "Simulation of an Epidemic")

**Theory.** When one bank becomes illiquid or insolvent, distress can spread
through interbank lending networks — a classic **financial contagion**
phenomenon (cf. Allen & Gale, 2000). We model it with a compartmental
(SIR-style) system over the *fraction of banks* in each state:

```
dS/dt = -beta * S * I     (Healthy banks becoming exposed/distressed)
dI/dt =  beta * S * I - gamma * I    (Distressed banks; gamma = resolution rate)
dR/dt =  gamma * I        (Resolved: recapitalized, merged, or wound down)
```

- `S` = fraction of banks that are healthy
- `I` = fraction currently distressed/illiquid
- `R` = fraction resolved (via bailout, merger, or orderly resolution)
- `beta` = interconnectedness / speed of contagion through interbank exposures
- `gamma` = speed of resolution (regulatory intervention, liquidity facilities)

This is solved exactly like the book's epidemic system in Listing 6.34, using
`solve_ivp` on a vector-valued ODE function.

In [ ]:
beta_contagion, gamma_resolution = 0.6, 0.15
I0 = 0.01                       # 1% of banks initially distressed
t_span_sir = (0, 60)
t_eval_sir = np.linspace(*t_span_sir, 400)
y0 = [1 - I0, I0, 0.0]          # [S0, I0, R0]

In [ ]:
# TODO 7a: Define contagion(t, y) returning [dS, dI, dR] for y=[S,I,R]:
#          dS = -beta_contagion*S*I
#          dI =  beta_contagion*S*I - gamma_resolution*I
#          dR =  gamma_resolution*I
def contagion(t, y):
    ...  # your code here

# TODO 7b: Solve with solve_ivp(contagion, t_span_sir, y0, t_eval=t_eval_sir)
sol_sir = None  # replace
S_t, I_t, R_t = None, None, None  # replace with sol_sir.y

# TODO 7c: Find the peak fraction distressed (np.argmax on I_t) and the
#          time at which it occurs.

# TODO 7d: Plot S(t), I(t), R(t) together (as percentages) over time.

# TODO 7e (bonus): Loop gamma_resolution over [0.10, 0.15, 0.30] (faster
#          regulatory intervention) and plot I(t) for each to see how a
#          faster resolution speed "flattens the curve" of contagion.

**Interpretation (answer here).** How does raising `gamma` (faster
regulatory resolution -- e.g. quicker liquidity facilities or resolution
authority action) change the *peak* fraction of distressed banks, even
though it doesn't change `beta` (interconnectedness) at all? Why does this
matter for how regulators design emergency lending facilities?

## 8. Wrap-Up

Write a short (150–250 word) summary in this cell covering:

- One insight from the bond-pricing/portfolio sections (market & credit risk).
- One insight from the ALM/systemic-risk sections (treasury & financial stability).
- Which SciPy technique from the book's Chapter 6 you found most directly
  transferable to banking data analytics, and why.

*(Your answer here.)*